# Index Table Preparation

In [2]:
import geopandas as gpd
import pandas as pd
import requests
import json
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit


# config
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

Pandas notes for me: 

gdf.loc[ROWS, COLUMNS] = "select rows by this value, and columns by this value"

new field:
gdf["new_field"] = (
    gdf["field_a"].astype(str)
    + "_"
    + gdf["field_b"].astype(str)
    + "_"
    + gdf["field_c"].astype(str)
)

### Load the Shapefile and give it a true unique ID before doing anything else to it. 

In [3]:
# this is the path to the original feature service, downloaded as a shapefile from here: 
# https://www.arcgis.com/home/item.html?id=f260e4065a384e7ebee05b23362f380f#overview
# Layer 36112 - https://gis5.uspatial.umn.edu/arcgis/rest/services/mhapo/MHAPO_dynamic/MapServer/0
original_index_path = "data/historic_airphoto_index/shapefiles/umn_airphoto_index_original.shp"

gdf_original = gpd.read_file(original_index_path)

print(gdf_original.crs)
# we're keeping this in web mercator for now. 

EPSG:3857


In [4]:
# make a unique identifier

gdf_original["source_id"] = range(1, len(gdf_original) + 1)

assert gdf_original["source_id"].is_unique
assert gdf_original["source_id"].notna().all()

In [ ]:
gdf_original

In [7]:
gdf_original.to_file(
    "data/historic_airphoto_index/umn_index_original_with_source_id.gpkg",
    driver="GPKG"
)

### Cleaning the original table

In [8]:
gdf_clean = gdf_original.copy()

#preview the columns
gdf_clean.columns

Index(['flight_cod', 'url_link', 'photo_id', 'lat', 'long_', 'year_',
       'objectid_1', 'in_fid', 'near_fid', 'near_dist', 'objectid_2',
       'statefp10', 'countyfp10', 'countyns10', 'geoid10', 'name10',
       'namelsad10', 'lsad10', 'classfp10', 'mtfcc10', 'csafp10', 'cbsafp10',
       'metdivfp10', 'funcstat10', 'aland10', 'awater10', 'intptlat10',
       'intptlon10', 'year_conve', 'decade', 'decade_dat', 'decade_str',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'geometry',
       'source_id'],
      dtype='str')

#### Cleaning the URLs
And looking for true duplicates...

In [9]:
# clean URL spaces

gdf_clean["url_link"] = gdf_clean["url_link"].str.strip()

In [40]:
exists = 'https://geo.lib.umn.edu/Olmsted/1940/WB-3A-115.jpg' in gdf_clean['url_link'].values

exists

False

In [ ]:
# identify duplicate URLs

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [ ]:
# identify duplicate URLs with unique photo_ids to be fixed

fake_duplicates = gdf_clean[
    gdf_clean["url_link"].duplicated(keep=False)
    & ~gdf_clean["photo_id"].duplicated(keep=False)]
fake_duplicates = fake_duplicates.sort_values(by=['flight_cod','url_link'])
fake_duplicates

In [ ]:
# fix olmsted county URLs

bad_url = "https://geo.lib.umn.edu/Olmsted/1940/WB-3A-101.jpg"

mask = gdf_clean["url_link"] == bad_url

gdf_clean.loc[mask, "url_link"] = (
    "https://geo.lib.umn.edu/Olmsted/1940/"
    + gdf_clean.loc[mask, "photo_id"]
    + ".jpg"
)

In [ ]:
check = gdf_clean[gdf_clean["flight_cod"] == "Olmsted"]

check

In [ ]:
# identify duplicate URLs again

duplicates = gdf_clean[gdf_clean["url_link"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','url_link'])
duplicates

In [17]:
# drop duplicates arbitrarily (the only thing that varies is the geometry and we're not using it.)
gdf_clean = gdf_clean.drop_duplicates(
    subset=["url_link"],
    keep="first"
).copy()

In [ ]:
# identify duplicate URLs again
# there should be none. 

duplicates = gdf_clean[gdf_clean["photo_id"].duplicated(keep=False)]
duplicates = duplicates.sort_values(by=['flight_cod','photo_id'])
duplicates

In [ ]:
print("Unique URLs:", gdf_clean["url_link"].is_unique)
print("Remaining records:", len(gdf_clean))

Unique URLs: True
Unique photo_ids: False
Remaining records: 115577


### Clean up other values

In [20]:
# convert year field to an integer rather than a float
# handles missing values
gdf_clean["year_"] = gdf_clean["year_"].astype("Int64")

In [21]:
# populate missing counties from the flight code field
gdf_clean["name10"] = gdf_clean["name10"].fillna(
    gdf_clean["flight_cod"]
)

In [22]:
# check for nulls
print(gdf_clean["name10"].isna().any())

print(gdf_clean["year_"].isna().any())

print(gdf_clean["photo_id"].isna().any())

print(gdf_clean["url_link"].isna().any())

False
False
False
False


### Adding filename identifier and validation

In [35]:
# add a new unique, human readable identifier field called "cty_year_id"
# it will be used for naming files
# it is a combination of the county, the year, and the photo ID. 

gdf_clean["cty_year_id"] = (
    gdf_clean["name10"].str.strip().str.lower().str.replace(" ", "_").str.replace('.', '', regex=False)
    + "_"
    + gdf_clean["year_"].astype("string")
    + "_"
    + gdf_clean["photo_id"].str.strip().str.lower().str.replace(" ", "_")
)

# add a number to any non-unique values produced by this
counts = gdf_clean.groupby("cty_year_id").cumcount()

gdf_clean["cty_year_id"] += counts.map(
    lambda n: f"_{n}" if n > 0 else ""
)

In [ ]:
# validate that the periods were handled correctly
gdf_clean_filtered = gdf_clean.loc[gdf_clean["name10"] == "St. Louis"]

gdf_clean_filtered

In [36]:
# validate that this new field is unique and contains no nulls
# should return 

print(f"All unique values:", gdf_clean['cty_year_id'].is_unique)
print(f"Has null values:", gdf_clean['cty_year_id'].isna().any())
print(f"Has spaces:", gdf_clean['cty_year_id'].str.contains(' ').any())

All unique values: True
Has null values: False
Has spaces: False


In [ ]:
# check it out
gdf_clean

In [ ]:
# save it
gdf_clean.to_file("data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg")

### Drop and reorder columns

In [76]:
# open cleaned table
path = "data/historic_airphoto_index/umn_index_cleaned_ogcolumns.gpkg"

gdf = gpd.read_file(path)

In [ ]:
gdf

In [77]:
# inspect
gdf.columns

Index(['flight_cod', 'url_link', 'photo_id', 'lat', 'long_', 'year_',
       'objectid_1', 'in_fid', 'near_fid', 'near_dist', 'objectid_2',
       'statefp10', 'countyfp10', 'countyns10', 'geoid10', 'name10',
       'namelsad10', 'lsad10', 'classfp10', 'mtfcc10', 'csafp10', 'cbsafp10',
       'metdivfp10', 'funcstat10', 'aland10', 'awater10', 'intptlat10',
       'intptlon10', 'year_conve', 'decade', 'decade_dat', 'decade_str',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'source_id',
       'cty_year_id', 'geometry'],
      dtype='str')

In [78]:
# rename
new_gdf = gdf.rename(columns={
    "url_link" : "url",
    "year_" : "year",
    "name10" : "county",
    'flight_cod' : 'flight_code'
})

In [87]:
# add
new_gdf["worldfile_json"] = None
new_gdf["h_extent"] = None
new_gdf["scale"] = None
new_gdf["dpi_x"] = None
new_gdf["width_px"] = None
new_gdf["status"] = None
new_gdf["gcp_count"] = None
new_gdf["confidence"] = None

In [88]:
new_gdf = new_gdf[['cty_year_id','status','county','year','photo_id','h_extent','gcp_count','confidence','scale','dpi_x','width_px','worldfile_json','url','flight_code','source_id','geometry']]

new_gdf

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,geometry
17231,aitkin_1939_bxv02017,None,Aitkin,1939,bxv02017,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxv/y1939/bxv02017.jpg,bxv,17233,POINT (-10383436.935 5806733.399)
17324,aitkin_1939_bxt03191,None,Aitkin,1939,bxt03191,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03191.jpg,bxt,17326,POINT (-10442747.814 5821533.312)
17325,aitkin_1939_bxt03192,None,Aitkin,1939,bxt03192,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03192.jpg,bxt,17327,POINT (-10440599.092 5821517.341)
17346,aitkin_1939_byd03058,None,Aitkin,1939,byd03058,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd03058.jpg,byd,17348,POINT (-10359703.23 5813165.04)
17348,aitkin_1939_byd04003,None,Aitkin,1939,byd04003,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd04003.jpg,byd,17350,POINT (-10358746.071 5818024.084)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9646,yellow_medicine_1991_napp_4112_148,None,Yellow Medicine,1991,napp_4112_148,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_148.jpg,napp,9647,POINT (-10655011.753 5597325.517)
9647,yellow_medicine_1991_napp_4112_149,None,Yellow Medicine,1991,napp_4112_149,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_149.jpg,napp,9648,POINT (-10655123.006 5602256.72)
9648,yellow_medicine_1991_napp_4113_036,None,Yellow Medicine,1991,napp_4113_036,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_036.jpg,napp,9649,POINT (-10648610.57 5601707.206)
9649,yellow_medicine_1991_napp_4113_037,None,Yellow Medicine,1991,napp_4113_037,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_037.jpg,napp,9650,POINT (-10648410.986 5596854.756)


In [92]:
new_gdf = new_gdf.sort_values(by=['county','year'], ascending=[True, True])

new_gdf

,cty_year_id,status,county,year,photo_id,h_extent,gcp_count,confidence,scale,dpi_x,width_px,worldfile_json,url,flight_code,source_id,geometry
17231,aitkin_1939_bxv02017,None,Aitkin,1939,bxv02017,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxv/y1939/bxv02017.jpg,bxv,17233,POINT (-10383436.935 5806733.399)
17324,aitkin_1939_bxt03191,None,Aitkin,1939,bxt03191,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03191.jpg,bxt,17326,POINT (-10442747.814 5821533.312)
17325,aitkin_1939_bxt03192,None,Aitkin,1939,bxt03192,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxt/y1939/bxt03192.jpg,bxt,17327,POINT (-10440599.092 5821517.341)
17346,aitkin_1939_byd03058,None,Aitkin,1939,byd03058,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd03058.jpg,byd,17348,POINT (-10359703.23 5813165.04)
17348,aitkin_1939_byd04003,None,Aitkin,1939,byd04003,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/byd/y1939/byd04003.jpg,byd,17350,POINT (-10358746.071 5818024.084)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9646,yellow_medicine_1991_napp_4112_148,None,Yellow Medicine,1991,napp_4112_148,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_148.jpg,napp,9647,POINT (-10655011.753 5597325.517)
9647,yellow_medicine_1991_napp_4112_149,None,Yellow Medicine,1991,napp_4112_149,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4112_149.jpg,napp,9648,POINT (-10655123.006 5602256.72)
9648,yellow_medicine_1991_napp_4113_036,None,Yellow Medicine,1991,napp_4113_036,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_036.jpg,napp,9649,POINT (-10648610.57 5601707.206)
9649,yellow_medicine_1991_napp_4113_037,None,Yellow Medicine,1991,napp_4113_037,None,None,None,None,None,None,None,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usgs/napp/y1991/napp_4113_037.jpg,napp,9650,POINT (-10648410.986 5596854.756)


In [93]:
new_gdf.to_file("data/historic_airphoto_index/index_table_cleaned_pre-json.gpkg")

### Get JPEG world file affine parameters, if they exist

In [94]:
gdf = gpd.read_file("data/historic_airphoto_index/index_table_cleaned_pre-json.gpkg")

In [95]:
def fetch_worldfile(jpg_url):
    """Retrieve a JGW and return its affine parameters as JSON."""

    if not isinstance(jpg_url, str) or not jpg_url:
        return None

    # Replace .jpg with .jgw
    jgw_url = jpg_url.removesuffix(".jpg") + ".jgw"

    try:
        response = requests.get(jgw_url, timeout=15)

        if response.status_code == 300:
            return None

        if response.status_code == 404:
            return None

        response.raise_for_status()

        # World-file parameter order: A, D, B, E, C, F
        values = [float(v) for v in response.text.split()]

        if len(values) != 6:
            raise ValueError("World file must contain exactly 6 values")

        parameters = dict(zip(
            ["A", "D", "B", "E", "C", "F"],
            values
        ))

        return json.dumps(parameters)

    except (requests.RequestException, ValueError) as e:
        print(f"Failed to retrieve {jgw_url}: {e}")
        return None

In [96]:
test_json = fetch_worldfile("https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/projects/stl/y1989/stl_028_107.jpg")

print(test_json)

{"A": 0.33, "D": 0.0, "B": 0.0, "E": -0.33, "C": 557254.625, "F": 5187792.5}


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import pandas as pd

# Fetch world files concurrently
urls = gdf["url"].tolist()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(tqdm(
        executor.map(fetch_worldfile, urls),
        total=len(urls),
        desc="Fetching world files"
    ))

# Store world file parameters
gdf["worldfile_json"] = results

# Check results
print(gdf["worldfile_json"].notna().value_counts())

# Save GeoPackage
gdf.to_file(
    "data/historic_airphoto_index/index_with_jsons.gpkg"
)

c:\Users\eu01251115\LocalProjects\batch_georef\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching world files:   0%|          | 93/115577 [00:08<2:45:50, 11.61it/s]

### Export

In [29]:
gdf.to_file("data/historic_airphoto_index/main_airphoto_index.gpkg")